# Pipeline walkthrough: one full self-healing cycle

This notebook runs every stage of the pipeline on `protocols/serial_dilution_v1.json`, importing the same
functions `cli.py` uses, so you can inspect what each module takes in and hands to the next.

```
protocol IR ─► Stage 1 verifier ─► Stage 2 compiler ─► Stage 3 ChatterBox simulation ─► PASSED
                    ▲                                              │ failure
                    │                                              ▼
                    └──── harness checks ◄── IR edits ◄── local model reads the diagnostic
```

| Section | Module | What you'll see |
|---|---|---|
| 1 | `protocols/*.json` | The protocol IR the scientist reviews |
| 2 | `pipeline/deck.py` | The PyLabRobot deck built from `config/plr_deck.json` |
| 3 | `pipeline/stage1_verifier.py` | Safety rules on the IR: a pass, and two deliberate failures |
| 4 | `pipeline/stage2_compiler.py` | The deterministic IR → PyLabRobot translation |
| 5 | `pipeline/stage3_simulator.py`, `pipeline/diagnostics.py` | ChatterBox dry run, the failing step, and the lab-concept diagnostic |
| 6 | `pipeline/self_heal.py` | The repair loop's checks, applied by hand with no model |
| 7 | `pipeline/self_heal.py` + Ollama | The real loop with `qwen2.5-coder:7b` (skipped if Ollama isn't running) |
| 8 | all | Independent re-check of the repaired protocol; saves its compiled PLR code |
| 9 | `pylabrobot.visualizer` | The healed protocol replayed in PyLabRobot's browser visualizer |

Sections 1–6 and 8 need no model and give the same results every run. Section 7 calls a local model.
Section 9 opens a browser tab.

In [1]:
import copy
import json
import sys
import urllib.request
from pathlib import Path
from typing import Optional

import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from pipeline.deck import build_deck, load_deck_config
from pipeline.stage1_verifier import Verifier
from pipeline.stage2_compiler import CompilationError, compile_protocol, translate_step
from pipeline.stage3_simulator import PLRSimulator
from pipeline.self_heal import OllamaBackend, SelfHealer, apply_edits, ir_diff

CONFIG = ROOT / "config"
PROTOCOL_PATH = ROOT / "protocols" / "serial_dilution_v1.json"
WORK = ROOT / "build" / "notebook"   # build/ is git-ignored
WORK.mkdir(parents=True, exist_ok=True)

protocol = yaml.safe_load(PROTOCOL_PATH.read_text())
deck_config = load_deck_config(str(CONFIG / "plr_deck.json"))
pd.set_option("display.max_colwidth", 120)
print(f"Loaded {PROTOCOL_PATH.name}: {len(protocol['steps'])} steps")

Loaded serial_dilution_v1.json: 68 steps


## 1. The protocol IR

The IR is the human-readable plan. It's the only thing the repair model is allowed to change, and what a
scientist reviews. Besides the steps, it records how much of each reagent the scientist loads before the run
(`initial_volumes_ul`).

In [2]:
print("Protocol:", protocol["protocol_name"])
print("Initial volumes (uL):", protocol["initial_volumes_ul"])
pd.DataFrame(protocol["dilution"]["columns"]).set_index("column").T

Protocol: 2-Fold Serial Dilution Protocol (12 columns, 1000 uM to 0.488 uM)
Initial volumes (uL): {'tube_15ml': 600, 'reservoir_50ml': 5500}


column,1,2,3,4,5,6,7,8,9,10,11,12
concentration_um,1000.0,500.0,250.0,125.0,62.5,31.25,15.625,7.8125,3.9062,1.9531,0.9766,0.4883


In [3]:
steps = pd.DataFrame(protocol["steps"]).set_index("step")
print("Steps per action:", steps["action"].value_counts().to_dict())
steps[["action", "labware", "column", "volume_ul", "description"]].head(16)

Steps per action: {'dispense': 24, 'aspirate': 16, 'tips_on': 14, 'tips_off': 14}


,action,labware,column,volume_ul,description
step,,,,,
1,tips_on,hamilton_tiprack_96,1.0,NaN,"Pick up 8 tips, rack 1 column 1, for stock fill"
2,aspirate,tube_15ml,1.0,100.0,Aspirate 100 uL stock (1000 uM) from 15 mL tube
3,dispense,corning_3595,1.0,100.0,Dispense 100 uL stock into plate column 1 (1000 uM)
4,tips_off,waste,NaN,NaN,Eject stock tips to waste
5,tips_on,hamilton_tiprack_96,2.0,NaN,"Pick up 8 tips, rack 1 column 2, for diluent"
6,aspirate,reservoir_50ml,1.0,200.0,Aspirate 200 uL diluent from reservoir for columns 2-5
7,dispense,corning_3595,2.0,50.0,Dispense 50 uL diluent into plate column 2
8,dispense,corning_3595,3.0,50.0,Dispense 50 uL diluent into plate column 3
9,dispense,corning_3595,4.0,50.0,Dispense 50 uL diluent into plate column 4


The plan has four stages: fill column 1 with stock (steps 1–4), add diluent to columns 2–12 (steps 5–20),
eleven column-to-column transfers with fresh tips each time, and a final discard from column 12.

**Step 2 has a physical problem the verifier can't see:** it lowers all 8 channels into one 15 mL tube at once.

## 2. The deck (`pipeline/deck.py`)

`config/plr_deck.json` maps every labware ID in the IR to a PyLabRobot resource and a carrier position. The
deck is fixed config, never model output: Stages 2–4 all build the same deck from the same file.

In [4]:
deck, res = build_deck(deck_config, protocol["initial_volumes_ul"])
rows = []
for lid, spec in deck_config["labware"].items():
    r = res[lid]
    rows.append({
        "IR labware ID": lid,
        "kind": spec["kind"],
        "PLR resource": type(r).__name__,
        "factory": spec.get("factory", "deck trash"),
        "carrier / slot": f"{spec['carrier']} / {spec['slot']}" if "carrier" in spec else "-",
        "width y (mm)": round(r.get_size_y(), 1),
        "starting volume (uL)": r.tracker.get_used_volume() if spec["kind"] == "container" else None,
    })
pd.DataFrame(rows)

,IR labware ID,kind,PLR resource,factory,carrier / slot,width y (mm),starting volume (uL)
0,hamilton_tiprack_96,tip_rack,TipRack,hamilton_96_tiprack_300uL_filter,tip_carrier / 0,82.6,NaN
1,hamilton_tiprack_96_b,tip_rack,TipRack,hamilton_96_tiprack_300uL_filter,tip_carrier / 1,82.6,NaN
2,corning_3595,plate,Plate,cor_96_wellplate_360uL_Fb,plate_carrier / 0,85.5,NaN
3,reservoir_50ml,container,Trough,hamilton_1_trough_60mL_Vb,trough_carrier / 0,90.0,5500.0
4,tube_15ml,container,Tube,cor_falcon_tube_15mL_Vb,tube_carrier / 0,17.0,600.0
5,waste,trash,Trash,deck trash,-,241.2,NaN


Note the tube's width (17 mm). Hamilton channels must be at least 9 mm apart, so only one channel fits inside it.

## 3. Stage 1: safety verifier (`pipeline/stage1_verifier.py`)

The verifier walks the IR step by step with its own model of the plate and the tips. It checks dead volume
(30 µL minimum), well capacity (330 µL), tip inventory per rack, mounted tips, and cross-contamination between
sources. It reads protocols from files, so this helper writes each variant to `build/notebook/`.

In [5]:
def verify(ir, name):
    path = WORK / f"{name}.json"
    path.write_text(json.dumps(ir, indent=2))
    v = Verifier(str(CONFIG / "labware.json"), str(path), str(CONFIG / "protocol_defaults.json"))
    return v.run_verification(), v

result, verifier = verify(protocol, "v1_original")
print(result)

{'status': 'PASSED', 'message': 'Successfully verified 68 declarative protocol steps.', 'errors': []}


The verifier's final plate model: **every well ends at 50 µL.** Each column receives 100 µL (stock, or 50 µL
diluent + 50 µL from the previous column) and passes 50 µL on to the next; column 12's 50 µL goes to waste.

⚠️ The IR's metadata says `dilution.final_volume_per_well_ul: 100`, which doesn't match what its steps do.
The pipeline only reads the steps, so this doesn't affect any result here, but the metadata (or the volumes)
should be corrected.

In [6]:
pd.DataFrame(verifier.plate_state, index=list("ABCDEFGH"), columns=range(1, 13))

,1,2,3,4,5,6,7,8,9,10,11,12
A,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0
B,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0
C,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0
D,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0
E,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0
F,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0
G,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0
H,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0


Two deliberately broken copies, to show what Stage 1 catches:

In [7]:
overflow = copy.deepcopy(protocol)
overflow["steps"][2]["volume_ul"] = 400.0        # step 3: dispense 400 uL of stock into column 1
print("Overflow:      ", verify(overflow, "broken_overflow")[0])

reused_tips = copy.deepcopy(protocol)
reused_tips["steps"][4]["column"] = 1            # step 5: pick up tips from rack column 1 again
print("Reused tips:   ", verify(reused_tips, "broken_tips")[0])

Overflow:       {'status': 'FAILED', 'error_code': 'VOLUMETRIC_OVERFLOW', 'message': 'Step 3: Volumetric overflow risk at Row 1, Col 1. Current: 0.0uL + 400.0uL exceeds max usable capacity (330uL from corning_3595).'}
Reused tips:    {'status': 'FAILED', 'error_code': 'TIP_DEPLETED', 'message': "Step 5: No tip available in rack 'hamilton_tiprack_96' at Row 1, Col 1."}


**Stage 1 passes the original v1 protocol**, but that doesn't make it runnable. The verifier has no model of
container geometry or of how much liquid is in the source tube. That's what Stage 3 is for.

## 4. Stage 2: compiler (`pipeline/stage2_compiler.py`)

A deterministic translator: the same IR always produces the same PyLabRobot code. Every IR step becomes one
statement, preceded by a `# step N` comment so Stage 3 can trace a runtime error back to its step.

In [8]:
module_code = compile_protocol(protocol, deck_config)
lines = module_code.splitlines()
print(f"{len(lines)} lines of generated code; first 16:\n")
print("\n".join(lines[:16]))

140 lines of generated code; first 16:

"""Compiled from protocol: 2-Fold Serial Dilution Protocol (12 columns, 1000 uM to 0.488 uM). Do not edit; recompile from the IR."""


async def run_protocol(lh, res):
    # step 1: Pick up 8 tips, rack 1 column 1, for stock fill
    await lh.pick_up_tips([res["hamilton_tiprack_96"].get_item("A1"), res["hamilton_tiprack_96"].get_item("B1"), res["hamilton_tiprack_96"].get_item("C1"), res["hamilton_tiprack_96"].get_item("D1"), res["hamilton_tiprack_96"].get_item("E1"), res["hamilton_tiprack_96"].get_item("F1"), res["hamilton_tiprack_96"].get_item("G1"), res["hamilton_tiprack_96"].get_item("H1")], use_channels=[0, 1, 2, 3, 4, 5, 6, 7])
    # step 2: Aspirate 100 uL stock (1000 uM) from 15 mL tube
    await lh.aspirate([res["tube_15ml"]] * 8, vols=[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], use_channels=[0, 1, 2, 3, 4, 5, 6, 7])
    # step 3: Dispense 100 uL stock into plate column 1 (1000 uM)
    await lh.dispense([res["corning_3595"].

How each kind of step translates. Plates and tip racks address one well per channel (channel *i* = row
`"ABCDEFGH"[i]`); a single container like a tube or trough is the same target for every channel.

In [9]:
kinds = {lid: spec["kind"] for lid, spec in deck_config["labware"].items()}
examples = {
    "tips_on, rack column 3, channels 0-1": {"step": 1, "action": "tips_on", "labware": "hamilton_tiprack_96", "column": 3, "channels": [0, 1]},
    "aspirate from a tube, channel 4":      {"step": 2, "action": "aspirate", "labware": "tube_15ml", "volume_ul": 100, "channels": [4]},
    "dispense into plate column 2":         {"step": 3, "action": "dispense", "labware": "corning_3595", "column": 2, "volume_ul": 50, "channels": [0, 1]},
    "tips_off":                             {"step": 4, "action": "tips_off", "labware": "waste"},
}
for label, step in examples.items():
    print(f"{label}:\n    {translate_step(step, kinds)}\n")

try:
    translate_step({"step": 9, "action": "aspirate", "labware": "corning_3595", "column": 1, "channels": [0]}, kinds)
except CompilationError as e:
    print("Malformed step ->", e)

tips_on, rack column 3, channels 0-1:
    await lh.pick_up_tips([res["hamilton_tiprack_96"].get_item("A3"), res["hamilton_tiprack_96"].get_item("B3")], use_channels=[0, 1])

aspirate from a tube, channel 4:
    await lh.aspirate([res["tube_15ml"]] * 1, vols=[100.0], use_channels=[4])

dispense into plate column 2:
    await lh.dispense([res["corning_3595"].get_item("A2"), res["corning_3595"].get_item("B2")], vols=[50.0, 50.0], use_channels=[0, 1])

tips_off:
    await lh.discard_tips()

Malformed step -> Step 9: aspirate needs 'volume_ul'.


## 5. Stage 3: simulation and diagnostics (`pipeline/stage3_simulator.py`, `pipeline/diagnostics.py`)

The compiled module runs against PyLabRobot's `LiquidHandlerChatterboxBackend` (a stand-in for the robot's
firmware) with PLR's tip and volume trackers on. If PLR raises, the interceptor:

1. finds the `# step N` comment above the failing line → the IR step,
2. snapshots the deck (mounted tips, container volumes, unused tip columns),
3. translates the exception into a lab-concept diagnostic with suggested IR edits.

In [10]:
simulator = PLRSimulator(deck_config)
sim = simulator.run(module_code, protocol)
print(sim.status, "-", sim.message)

FAILED - Step 2: CHANNEL_SPACING (ValueError: Resource is too small to space channels.)


In [11]:
d = sim.diagnostic
print("Error code:  ", d["error_code"])
print("Raw error:   ", d["raw_error"])
print("Failing step:", d["step_ir"])
print("\nExplanation:\n ", d["explanation"])
print("\nSuggested fixes:")
for fix in d["suggested_fixes"]:
    print("  -", fix)

Error code:   CHANNEL_SPACING
Raw error:    ValueError: Resource is too small to space channels.
Failing step: {'step': 2, 'action': 'aspirate', 'labware': 'tube_15ml', 'column': 1, 'channels': [0, 1, 2, 3, 4, 5, 6, 7], 'volume_ul': 100.0, 'description': 'Aspirate 100 uL stock (1000 uM) from 15 mL tube'}

Explanation:
  Step 2 sends 8 channels into 'tube_15ml' at the same time. Hamilton channels must stay at least 9 mm apart, and 'tube_15ml' is only 17 mm wide, so about 1 channel(s) fit in it at once. The robot physically cannot lower all 8 channels into it together.

Suggested fixes:
  - Replace step 2 with 8 steps of the same action, each using a single channel (channels [0], then [1], ... [7]) and the same labware and volume_ul.
  - Or move this reagent to a container wide enough for 8 channels (a trough), if one is on the deck.


In [12]:
state = d["deck_state"]
print("Tips mounted per channel:", state["tips_mounted"])
print("Container volumes (uL):  ", state["container_volumes_ul"])
print("Unused tip columns:      ", state["unused_tip_columns"])

Tips mounted per channel: {0: True, 1: True, 2: True, 3: True, 4: True, 5: True, 6: True, 7: True}
Container volumes (uL):   {'reservoir_50ml': 5500, 'tube_15ml': 600}
Unused tip columns:       {'hamilton_tiprack_96': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], 'hamilton_tiprack_96_b': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]}


What ChatterBox printed before the failure. These are the commands PLR would have sent to the robot:

In [13]:
print(sim.log[-1500:])

Setting up the liquid handler.
Picking up tips:
pip#  resource             offset           tip type     max volume (µL)  fitting depth (mm)   tip length (mm)  filter    
  p0: hamilton_tiprack_96_tipspot_A1 0,0,0            HamiltonTip  360              8                    59.9             Yes       
  p1: hamilton_tiprack_96_tipspot_B1 0,0,0            HamiltonTip  360              8                    59.9             Yes       
  p2: hamilton_tiprack_96_tipspot_C1 0,0,0            HamiltonTip  360              8                    59.9             Yes       
  p3: hamilton_tiprack_96_tipspot_D1 0,0,0            HamiltonTip  360              8                    59.9             Yes       
  p4: hamilton_tiprack_96_tipspot_E1 0,0,0            HamiltonTip  360              8                    59.9             Yes       
  p5: hamilton_tiprack_96_tipspot_F1 0,0,0            HamiltonTip  360              8                    59.9             Yes       
  p6: hamilton_tiprack_96_tipsp

## 6. The repair loop's checks, applied by hand (`pipeline/self_heal.py`)

`SelfHealer.evaluate()` runs Stages 1–3 and stops at the first failure. In the real loop, a model reads the
feedback and proposes edits. Here we play the model ourselves, including the bad edits a small model actually
made, to show what the harness accepts and rejects.

The model can only reply with two kinds of edits:
- `replace_step`: replace the **failing** step with one or more steps
- `set_initial_volume`: change how much reagent the scientist loads

Every proposed edit must pass three checks before it touches the IR:

| Check | Rejects |
|---|---|
| **Step scope** | Edits to any step other than the one that failed |
| **Compiles** | New steps that Stage 2 can't translate (missing volume, unknown labware, ...) |
| **Intent guard** | Anything that changes how much liquid each channel moves from or to each place |

In [14]:
healer = SelfHealer(
    deck_config=deck_config,
    labware_path=str(CONFIG / "labware.json"),
    protocol_defaults_path=str(CONFIG / "protocol_defaults.json"),
    backend=OllamaBackend(),          # not called in this section
    work_dir=str(WORK / "manual"),
)
baseline = healer._transfers(protocol)   # each channel's total aspirate/dispense volume per location

def propose(ir, edits, failing_step) -> Optional[dict]:
    # Run an edit through the same checks the real loop uses; return the new IR or None if rejected.
    new_ir, errors = apply_edits(ir, edits, healer.labware_kinds, allowed_step=failing_step)
    if not errors:
        errors = healer._intent_violations(baseline, healer._transfers(new_ir))
    if errors:
        print("REJECTED:")
        for e in errors[:4]:
            print("  -", e)
        return None
    print(f"ACCEPTED: protocol now has {len(new_ir['steps'])} steps")
    return new_ir

gate = healer.evaluate(protocol, 0)
print(f"[{gate.stage}] failing step {gate.step}\n")
print("This is the exact prompt the model receives (steps truncated):\n")
prompt = healer._prompt(protocol, gate, [])
print(prompt[:900], "\n   ...\n", prompt[-900:])

[simulate] failing step 2

This is the exact prompt the model receives (steps truncated):

Deck labware:
- hamilton_tiprack_96 (tip_rack)
- hamilton_tiprack_96_b (tip_rack)
- corning_3595 (plate)
- reservoir_50ml (container)
- tube_15ml (container)
- waste (trash)

Initial volumes loaded by the scientist (uL): {"tube_15ml": 600, "reservoir_50ml": 5500}

Protocol steps:
{"step": 1, "action": "tips_on", "labware": "hamilton_tiprack_96", "column": 1, "channels": [0, 1, 2, 3, 4, 5, 6, 7]}
{"step": 2, "action": "aspirate", "labware": "tube_15ml", "column": 1, "volume_ul": 100.0, "channels": [0, 1, 2, 3, 4, 5, 6, 7]}
{"step": 3, "action": "dispense", "labware": "corning_3595", "column": 1, "volume_ul": 100.0, "channels": [0, 1, 2, 3, 4, 5, 6, 7]}
{"step": 4, "action": "tips_off", "labware": "waste", "channels": [0, 1, 2, 3, 4, 5, 6, 7]}
{"step": 5, "action": "tips_on", "labware": "hamilton_tiprack_96", "column": 2, "channels": [0, 1, 2, 3, 4, 5, 6, 7]}
{"step": 6, "action": "aspirat 
   ...


**Round 1: fix the channel-spacing failure at step 2.** First, two bad attempts:

In [15]:
single_channel_aspirates = [
    {"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [ch],
     "description": f"Aspirate 100 uL stock with channel {ch}"}
    for ch in range(8)
]

print("Edit touching a step that didn't fail:")
propose(protocol, [{"op": "replace_step", "step": 30, "new_steps": []}], gate.step)

print("\nSplit that forgets the volumes (what qwen did in run 3):")
no_volume = [{k: v for k, v in s.items() if k != "volume_ul"} for s in single_channel_aspirates]
propose(protocol, [{"op": "replace_step", "step": 2, "new_steps": no_volume}], gate.step)

Edit touching a step that didn't fail:
REJECTED:
  - Only the failing step 2 may be replaced this round, not step 30.

Split that forgets the volumes (what qwen did in run 3):
REJECTED:
  - Step 2 (new step 1): aspirate needs 'volume_ul'.
  - Step 2 (new step 2): aspirate needs 'volume_ul'.
  - Step 2 (new step 3): aspirate needs 'volume_ul'.
  - Step 2 (new step 4): aspirate needs 'volume_ul'.


In [16]:
print("Correct split: 8 single-channel aspirates of 100 uL each")
round1 = propose(protocol, [{"op": "replace_step", "step": 2, "new_steps": single_channel_aspirates}], gate.step)
assert round1 is not None

gate = healer.evaluate(round1, 1)
print(f"\n[{gate.stage}] {'PASSED' if gate.passed else 'FAILED'}: {gate.summary}\n")
print(gate.feedback)

Correct split: 8 single-channel aspirates of 100 uL each
ACCEPTED: protocol now has 75 steps



[simulate] FAILED: Step 8: INSUFFICIENT_LIQUID (TooLittleLiquidError: Not enough liquid in container: 100.0uL > 0.0uL.)

Simulation failed at step 8 (INSUFFICIENT_LIQUID).
Step 8 tries to aspirate 100.0 uL per channel from 'tube_15ml', but it only holds 0.0 uL at this point. The plan loads 600 uL into 'tube_15ml' but aspirates 800 uL from it in total, so it runs dry before the protocol finishes.
Suggested fixes:
- Have the scientist load more reagent (needed 800 uL plus a 10% margin): {"op": "set_initial_volume", "labware": "tube_15ml", "volume_ul": 900}. This keeps every step unchanged.
- Lower volume_ul in step 8 so the source keeps enough liquid.


**Round 2: the next real failure.** With step 2 fixed, the simulation gets further and finds that the tube runs
dry: 600 µL loaded, 800 µL needed. Again, the bad edit first. This is the one qwen actually proposed in
run 5, a single 8-channel aspirate of 60 µL:

In [17]:
qwen_edit = [{"op": "replace_step", "step": gate.step, "new_steps": [
    {"action": "aspirate", "labware": "tube_15ml", "volume_ul": 60.0, "channels": list(range(8)),
     "description": "Reduced volume per channel to 60 uL to prevent running dry"}]}]
propose(round1, qwen_edit, gate.step)

REJECTED:
  - channel 0 aspirate at tube_15ml: 100 -> 160 uL
  - channel 1 aspirate at tube_15ml: 100 -> 160 uL
  - channel 2 aspirate at tube_15ml: 100 -> 160 uL
  - channel 3 aspirate at tube_15ml: 100 -> 160 uL


In [18]:
print("Load more stock instead (the diagnostic's first suggestion):")
round2 = propose(round1, [{"op": "set_initial_volume", "labware": "tube_15ml", "volume_ul": 900}], gate.step)
assert round2 is not None

gate = healer.evaluate(round2, 2)
print(f"\n[{gate.stage}] {'PASSED' if gate.passed else 'FAILED'}: {gate.summary}")

Load more stock instead (the diagnostic's first suggestion):
ACCEPTED: protocol now has 75 steps



[simulate] PASSED: All 75 steps ran in ChatterBox with tip and volume tracking.


What a scientist would review: only the real changes, with renumbering ignored.

In [19]:
print(ir_diff(protocol, round2))

--- original
+++ healed
@@ -1,4 +1,11 @@
-initial_volumes_ul: {"tube_15ml": 600, "reservoir_50ml": 5500}
+initial_volumes_ul: {"tube_15ml": 900, "reservoir_50ml": 5500}
 {"action": "tips_on", "labware": "hamilton_tiprack_96", "column": 1, "channels": [0, 1, 2, 3, 4, 5, 6, 7]}
-{"action": "aspirate", "labware": "tube_15ml", "column": 1, "volume_ul": 100.0, "channels": [0, 1, 2, 3, 4, 5, 6, 7]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [0]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [1]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [2]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [3]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [4]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [5]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [6]}
+{"action

## 7. The real loop with a local model

The same cycle, with `qwen2.5-coder:7b` choosing the edits instead of us. It needs Ollama running with the model
pulled (`ollama pull qwen2.5-coder:7b`); otherwise this section is skipped. Results can vary from run to run, and
a run may end with `GAVE_UP`. That's a safe outcome: nothing unverified gets through.

In [20]:
MODEL = "qwen2.5-coder:7b"

def ollama_has(model):
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3) as r:
            return any(m["name"] == model for m in json.loads(r.read())["models"])
    except OSError:
        return False

heal_result = None
if ollama_has(MODEL):
    model_healer = SelfHealer(
        deck_config=deck_config,
        labware_path=str(CONFIG / "labware.json"),
        protocol_defaults_path=str(CONFIG / "protocol_defaults.json"),
        backend=OllamaBackend(model=MODEL),
        work_dir=str(WORK / "heal"),
    )
    heal_result = model_healer.heal(protocol, max_rounds=5)
    print("\nResult:", heal_result.status)
else:
    print(f"Ollama or {MODEL} not available; skipping the live run.")

  round 0: simulate FAILED - Step 2: CHANNEL_SPACING (ValueError: Resource is too small to space channels.)


    model: The failure is due to attempting to aspirate 8 channels into a tube that is too narrow. To fix this, we need to break step 2 into 8 separate steps, each using a single channel.
    applied: CHANNEL_SPACING ("Step 2: CHANNEL_SPACING (ValueError: Resource is too small to space channels.)") was fixed: replaced the failing step with 8 step(s)
  round 1: simulate FAILED - Step 8: INSUFFICIENT_LIQUID (TooLittleLiquidError: Not enough liquid in container: 100.0uL > 0.0uL.)


    model: The protocol fails because it attempts to aspirate 100.0 uL per channel from 'tube_15ml', but the tube only holds 600 uL. To fix this, we need to either load more reagent or reduce the volume per channel. Since the protocol must remain the same, we will reduce the volume per channel to 60 uL to ensu
    edits rejected: ["the edit changes how much liquid is moved (channel 0 aspirate at tube_15ml: 100 -> 160 uL; channel 1 aspirate at tube_15ml: 100 -> 160 uL; channel 2 aspirate at tube_15ml: 100 -> 160 uL; ...). A repair must keep every channel's total aspirated and dispensed volumes the same; load more reagent instead of moving less"]
  round 2: simulate FAILED - Step 8: INSUFFICIENT_LIQUID (TooLittleLiquidError: Not enough liquid in container: 100.0uL > 0.0uL.)


    model: The protocol fails because it attempts to aspirate 100 uL from 'tube_15ml' in each of the 8 channels, totaling 800 uL, but only 600 uL is initially loaded. To fix this, we need to either load more reagent or reduce the volume aspirated per channel. Since the protocol must keep the same total volumes
    applied: INSUFFICIENT_LIQUID ("Step 8: INSUFFICIENT_LIQUID (TooLittleLiquidError: Not enough liquid in container: 100.0uL > 0.0uL.)") was fixed: set the initial volume of tube_15ml to 900 uL
  round 3: simulate PASSED - All 75 steps ran in ChatterBox with tip and volume tracking.

Result: HEALED


In [21]:
if heal_result is not None:
    table = pd.DataFrame([
        {
            "round": r["round"],
            "stage": r["stage"],
            "passed": r["passed"],
            "failure": r["result"][:70],
            "model edit": "; ".join(
                f"{e['op']} step {e.get('step')} -> {len(e.get('new_steps', []))} step(s)" if e["op"] == "replace_step"
                else f"{e['op']} {e.get('labware')} = {e.get('volume_ul')}"
                for e in r.get("edits", [])
            ),
            "rejected because": (r.get("edit_errors") or [""])[0][:90],
        }
        for r in heal_result.rounds
    ])
    display(table)
    if heal_result.status == "HEALED":
        print(ir_diff(protocol, heal_result.protocol))

,round,stage,passed,failure,model edit,rejected because
0,0,simulate,False,Step 2: CHANNEL_SPACING (ValueError: Resource is too small to space ch,replace_step step 2 -> 8 step(s),
1,1,simulate,False,Step 8: INSUFFICIENT_LIQUID (TooLittleLiquidError: Not enough liquid i,replace_step step 8 -> 1 step(s),the edit changes how much liquid is moved (channel 0 aspirate at tube_15ml: 100 -> 160 uL;
2,2,simulate,False,Step 8: INSUFFICIENT_LIQUID (TooLittleLiquidError: Not enough liquid i,set_initial_volume tube_15ml = 900,
3,3,simulate,True,All 75 steps ran in ChatterBox with tip and volume tracking.,,


--- original
+++ healed
@@ -1,4 +1,11 @@
-initial_volumes_ul: {"tube_15ml": 600, "reservoir_50ml": 5500}
+initial_volumes_ul: {"tube_15ml": 900, "reservoir_50ml": 5500}
 {"action": "tips_on", "labware": "hamilton_tiprack_96", "column": 1, "channels": [0, 1, 2, 3, 4, 5, 6, 7]}
-{"action": "aspirate", "labware": "tube_15ml", "column": 1, "volume_ul": 100.0, "channels": [0, 1, 2, 3, 4, 5, 6, 7]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [0]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [1]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [2]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [3]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [4]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [5]}
+{"action": "aspirate", "labware": "tube_15ml", "volume_ul": 100.0, "channels": [6]}
+{"action

## 8. Independent re-check

Whatever produced the repaired IR, it goes through all three stages again from scratch, with no model involved.
This uses the model's result if section 7 healed the protocol, and otherwise the hand-made repair from section 6.

In [22]:
healed = heal_result.protocol if heal_result is not None and heal_result.status == "HEALED" else round2
print("Re-checking:", "model's repair" if healed is not round2 else "hand-made repair")

print("Stage 1:", verify(healed, "healed_recheck")[0]["status"])
healed_code = compile_protocol(healed, deck_config)
final = PLRSimulator(deck_config).run(healed_code, healed)
print("Stage 3:", final.status, "-", final.message)
print("\nLeft over after the run (uL):", final.final_state["container_volumes_ul"])
print("Unused tip columns:         ", final.final_state["unused_tip_columns"])

healed_plr_path = WORK / "serial_dilution_v1_healed_plr.py"
healed_plr_path.write_text(healed_code)
print("\nCompiled healed PLR protocol saved to:", healed_plr_path.relative_to(ROOT))

Re-checking: model's repair
Stage 1: PASSED


Stage 3: PASSED - All 75 steps ran in ChatterBox with tip and volume tracking.

Left over after the run (uL): {'reservoir_50ml': 1100.0, 'tube_15ml': 100.0}
Unused tip columns:          {'hamilton_tiprack_96': [3, 4, 5, 6, 7, 8, 9, 10, 11, 12], 'hamilton_tiprack_96_b': []}

Compiled healed PLR protocol saved to: build/notebook/serial_dilution_v1_healed_plr.py


## 9. Watch the healed protocol in PyLabRobot's visualizer

PyLabRobot ships a browser visualizer (`pylabrobot.visualizer.Visualizer`) that draws the deck and updates
live as tips are picked up and liquid moves. This section replays the healed protocol on a fresh ChatterBox
deck (the same deck, code, and trackers as Stage 3), pausing briefly after each step so you can follow it.

Running the next cell opens a browser tab at `http://127.0.0.1:1337`. If no tab opens, visit that address yourself.
The replay starts once the page has connected (or after 20 seconds).

In [23]:
import asyncio
import contextlib
import inspect
import io
import os

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import LiquidHandlerChatterboxBackend
from pylabrobot.resources import set_tip_tracking, set_volume_tracking
from pylabrobot.visualizer import Visualizer

HEADLESS = os.environ.get("NOTEBOOK_HEADLESS") == "1"   # set when the notebook is executed non-interactively
STEP_DELAY_S = 0.4                                       # pause after each step during the replay

set_tip_tracking(True)
set_volume_tracking(True)
vis_deck, vis_res = build_deck(deck_config, healed["initial_volumes_ul"])
vis_lh = LiquidHandler(backend=LiquidHandlerChatterboxBackend(num_channels=8), deck=vis_deck)
await vis_lh.setup()

visualizer = Visualizer(resource=vis_lh, open_browser=not HEADLESS, name="Healed serial_dilution_v1")
await visualizer.setup()
print(f"Visualizer running at http://{visualizer.host}:{visualizer.fs_port}")

if not HEADLESS:
    for _ in range(100):
        if visualizer.has_connection():
            break
        await asyncio.sleep(0.2)
    print("Browser connected." if visualizer.has_connection() else "No browser connected yet; replaying anyway.")

Setting up the liquid handler.
Websocket server started at http://127.0.0.1:2122
File server started at http://127.0.0.1:1338 . Open this URL in your browser.
Visualizer running at http://127.0.0.1:1338


In [24]:
class Paced:
    # Wraps the LiquidHandler so every call the protocol makes pauses afterwards, slow enough to watch.
    def __init__(self, lh, delay_s):
        self._lh, self._delay_s = lh, delay_s

    def __getattr__(self, name):
        method = getattr(self._lh, name)

        async def paced(*args, **kwargs):
            result = method(*args, **kwargs)
            if inspect.isawaitable(result):
                result = await result
            await asyncio.sleep(self._delay_s)
            return result
        return paced

namespace = {}
exec(compile(healed_code, "<healed_plr>", "exec"), namespace)
with contextlib.redirect_stdout(io.StringIO()):   # hide ChatterBox's per-command printout
    await namespace["run_protocol"](Paced(vis_lh, 0 if HEADLESS else STEP_DELAY_S), vis_res)

plate = vis_res["corning_3595"]
print(f"Replay finished: {len(healed['steps'])} steps.")
print("Plate row A volumes (uL):", [plate.get_item(f"A{c}").tracker.get_used_volume() for c in range(1, 13)])

Replay finished: 75 steps.
Plate row A volumes (uL): [50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0]


In [25]:
# The visualizer keeps running so you can look around. Set STOP_NOW = True and re-run this cell to close it.
STOP_NOW = HEADLESS
if STOP_NOW:
    await visualizer.stop()
    await vis_lh.stop()
    print("Visualizer stopped.")
else:
    print(f"Visualizer still running at http://{visualizer.host}:{visualizer.fs_port}")

Stopping the liquid handler.
Visualizer stopped.


## How the pieces fit

| Pivot step | Where it lives | Status |
|---|---|---|
| 1. Intent → human-readable JSON IR | `protocols/*.json` | Written by hand for now; no NL → IR generation yet |
| Scientist audit | IR files, `ir_diff` | The repaired IR is saved separately for review, never applied automatically |
| 2. In-silico dry run | Stage 1 verifier + Stage 2 compiler + Stage 3 ChatterBox | Working |
| 3. Domain error translation | `pipeline/diagnostics.py` | Covers channel spacing, liquid shortage, tip/container overflow, tip errors, contamination |
| 4. Closed-loop self-correction | `pipeline/self_heal.py` | Working with a local 7B model; one documented run in `docs/self_healing_run_v1.md` |
| Physical run | `pipeline/stage4_executor.py` | Stub |